# DPG 2.0 Paper Reproducibility Notebook

This notebook is organized as a paper-facing analysis pipeline for the current DPG local-explanation study.

It covers:
- data and artifact resolution
- best DPG 2.0 configurations across the 15 datasets
- baseline comparison against SHAP, LIME, ICE, Anchors, and the tree-native baseline when available
- quantitative rankings and paired statistical comparisons
- structural faithfulness and disagreement analysis
- semantic-faithfulness analysis: sufficiency, comprehensiveness, and critical-branch flips
- Phase C case studies, tables, and images
- auto-generated claim checks aligned with the current results


In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd

try:
    import scipy.stats as stats
except Exception:
    stats = None

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import Markdown, display, Image

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid', context='talk')


def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / 'experiments_local_explanation').exists() and (candidate / 'dpg').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root from the current working directory.')


repo_root = find_repo_root(Path.cwd())
analysis_root = repo_root / 'experiments_local_explanation' / 'experiment_dpg2_next_phase' / '_analysis'
baseline_root = repo_root / 'experiments_local_explanation' / 'results_baselines_by_dataset'

comparison_candidates = [
    analysis_root / 'comparison_story_phase_c_critical_mix',
    analysis_root / 'comparison_story_phase_c',
    analysis_root / 'comparison_story_local_dpg',
    analysis_root / 'comparison_story_fast',
    analysis_root / 'comparison_story',
]
comparison_dir = next((p for p in comparison_candidates if (p / 'consolidated_comparison_summary.csv').exists()), None)
if comparison_dir is None:
    raise FileNotFoundError('No comparison-story directory with consolidated results was found.')

semantic_candidates = [
    analysis_root / 'semantic_faithfulness',
    analysis_root / 'semantic_case_reports',
]
semantic_dir = next((p for p in semantic_candidates if p.exists()), None)

print('repo_root      :', repo_root)
print('analysis_root  :', analysis_root)
print('baseline_root  :', baseline_root)
print('comparison_dir :', comparison_dir)
print('semantic_dir   :', semantic_dir)


In [ ]:
def read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, low_memory=False) if path.exists() else pd.DataFrame()


def collect_baseline_summaries(root: Path) -> pd.DataFrame:
    frames = []
    if root.exists():
        for ds_dir in sorted(p for p in root.iterdir() if p.is_dir()):
            path = ds_dir / 'summary_baselines.csv'
            if path.exists():
                df = pd.read_csv(path, low_memory=False)
                if not df.empty:
                    df['dataset'] = ds_dir.name
                    frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def best_baseline_configs(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    return (
        df.sort_values(
            ['dataset', 'method', 'local_matches_model_rate', 'local_accuracy', 'avg_score_margin_pred_vs_competitor'],
            ascending=[True, True, False, False, False],
        )
        .drop_duplicates(subset=['dataset', 'method'], keep='first')
        .reset_index(drop=True)
    )


def top_by_method(df: pd.DataFrame, metric_cols: list[str]) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    agg = (
        df.groupby('method', as_index=False)[metric_cols]
        .mean(numeric_only=True)
        .sort_values(metric_cols, ascending=[False] * len(metric_cols))
        .reset_index(drop=True)
    )
    agg.insert(0, 'rank', np.arange(1, len(agg) + 1))
    return agg


def paired_wilcoxon(df: pd.DataFrame, left: str, right: str, value_col: str) -> dict[str, float | int | str]:
    if stats is None:
        return {'test': 'wilcoxon', 'left': left, 'right': right, 'metric': value_col, 'n': 0, 'statistic': np.nan, 'pvalue': np.nan}
    pivot = df.pivot(index='dataset', columns='method', values=value_col)
    if left not in pivot.columns or right not in pivot.columns:
        return {'test': 'wilcoxon', 'left': left, 'right': right, 'metric': value_col, 'n': 0, 'statistic': np.nan, 'pvalue': np.nan}
    paired = pivot[[left, right]].dropna()
    if len(paired) < 3:
        return {'test': 'wilcoxon', 'left': left, 'right': right, 'metric': value_col, 'n': int(len(paired)), 'statistic': np.nan, 'pvalue': np.nan}
    stat = stats.wilcoxon(paired[left], paired[right], alternative='two-sided', zero_method='wilcox')
    return {
        'test': 'wilcoxon',
        'left': left,
        'right': right,
        'metric': value_col,
        'n': int(len(paired)),
        'statistic': float(stat.statistic),
        'pvalue': float(stat.pvalue),
    }


def friedman_table(df: pd.DataFrame, methods: list[str], value_col: str) -> dict[str, object]:
    if stats is None:
        return {'metric': value_col, 'methods': methods, 'n_datasets': 0, 'statistic': np.nan, 'pvalue': np.nan}
    pivot = df.pivot(index='dataset', columns='method', values=value_col)
    available = [m for m in methods if m in pivot.columns]
    if len(available) < 3:
        return {'metric': value_col, 'methods': available, 'n_datasets': 0, 'statistic': np.nan, 'pvalue': np.nan}
    pivot = pivot[available].dropna()
    if len(pivot) < 3:
        return {'metric': value_col, 'methods': available, 'n_datasets': int(len(pivot)), 'statistic': np.nan, 'pvalue': np.nan}
    stat = stats.friedmanchisquare(*[pivot[m].values for m in available])
    return {'metric': value_col, 'methods': available, 'n_datasets': int(len(pivot)), 'statistic': float(stat.statistic), 'pvalue': float(stat.pvalue)}


combined_summary = read_csv(analysis_root / 'combined_summary.csv')
best_configs = read_csv(analysis_root / 'best_configs.csv')
best_per_sample = read_csv(analysis_root / 'best_per_sample.csv')
best_cohort_summary = read_csv(analysis_root / 'best_cohort_summary.csv')
best_agreement_summary = read_csv(analysis_root / 'best_agreement_summary.csv')
comparison_summary = read_csv(comparison_dir / 'consolidated_comparison_summary.csv')
cohort_effect_sizes = read_csv(comparison_dir / 'cohort_effect_sizes.csv')
case_study_metadata = read_csv(comparison_dir / 'case_study_metadata.csv')
final_paper_subset = read_csv(comparison_dir / 'final_paper_subset.csv') if (comparison_dir / 'final_paper_subset.csv').exists() else pd.DataFrame()

baseline_summary = collect_baseline_summaries(baseline_root)
baseline_best = best_baseline_configs(baseline_summary)

semantic_summary = read_csv(semantic_dir / 'semantic_faithfulness_summary.csv') if semantic_dir else pd.DataFrame()
semantic_per_sample = read_csv(semantic_dir / 'semantic_faithfulness_per_sample.csv') if semantic_dir else pd.DataFrame()
critical_branch_flip_summary = read_csv(semantic_dir / 'critical_branch_flip_summary.csv') if semantic_dir else pd.DataFrame()
critical_case_reports = read_csv(semantic_dir / 'critical_case_reports.csv') if semantic_dir else pd.DataFrame()

print('combined_summary      :', combined_summary.shape)
print('best_configs         :', best_configs.shape)
print('baseline_summary     :', baseline_summary.shape)
print('comparison_summary   :', comparison_summary.shape)
print('case_study_metadata  :', case_study_metadata.shape)
print('semantic_summary     :', semantic_summary.shape)
print('critical_case_reports:', critical_case_reports.shape)


## 1. Data Audit And Coverage

In [ ]:
coverage = pd.DataFrame({
    'source': ['best_configs', 'baseline_best', 'comparison_summary', 'case_studies', 'semantic_summary'],
    'datasets': [
        best_configs['dataset'].nunique() if not best_configs.empty else 0,
        baseline_best['dataset'].nunique() if not baseline_best.empty else 0,
        int(comparison_summary['datasets'].max()) if not comparison_summary.empty and 'datasets' in comparison_summary.columns else 0,
        case_study_metadata['dataset'].nunique() if not case_study_metadata.empty else 0,
        semantic_per_sample['dataset'].nunique() if not semantic_per_sample.empty else 0,
    ],
    'rows': [len(best_configs), len(baseline_best), len(comparison_summary), len(case_study_metadata), len(semantic_per_sample)],
})
display(coverage)

display(Markdown('### Best DPG Configurations'))
display(best_configs[['dataset', 'graph_construction_mode', 'config_id', 'local_matches_model_rate', 'local_accuracy', 'avg_edge_precision', 'avg_recombination_rate', 'avg_explanation_confidence']].sort_values(['dataset', 'graph_construction_mode']).reset_index(drop=True))

display(Markdown('### Best Baseline Configurations'))
baseline_cols = [c for c in ['dataset', 'method', 'config_id', 'local_matches_model_rate', 'local_accuracy', 'avg_score_margin_pred_vs_competitor', 'avg_anchor_precision', 'avg_anchor_coverage'] if c in baseline_best.columns]
display(baseline_best[baseline_cols].sort_values(['dataset', 'method']).reset_index(drop=True))


## 2. Main Quantitative Results

In [ ]:
best_dpg_table = best_configs[['dataset', 'graph_construction_mode', 'local_matches_model_rate', 'local_accuracy', 'avg_evidence_margin_pred_vs_competitor', 'avg_edge_precision', 'avg_edge_recall', 'avg_recombination_rate', 'avg_explanation_confidence']].copy()
best_dpg_table = best_dpg_table.rename(columns={'graph_construction_mode': 'method', 'avg_evidence_margin_pred_vs_competitor': 'margin', 'local_matches_model_rate': 'fidelity', 'local_accuracy': 'local_acc', 'avg_edge_precision': 'edge_precision', 'avg_edge_recall': 'edge_recall', 'avg_recombination_rate': 'recombination', 'avg_explanation_confidence': 'explanation_confidence'})

best_baseline_table = baseline_best[['dataset', 'method', 'local_matches_model_rate', 'local_accuracy', 'avg_score_margin_pred_vs_competitor']].copy()
best_baseline_table = best_baseline_table.rename(columns={'local_matches_model_rate': 'fidelity', 'local_accuracy': 'local_acc', 'avg_score_margin_pred_vs_competitor': 'margin'})

paired_quant = pd.concat([best_dpg_table, best_baseline_table], ignore_index=True, sort=False)

display(Markdown('### Consolidated Comparison Summary'))
display(comparison_summary.sort_values(['mean_fidelity', 'mean_local_accuracy', 'mean_margin'], ascending=False).reset_index(drop=True))

display(Markdown('### Mean Rankings Across Methods'))
ranking = top_by_method(paired_quant, ['fidelity', 'local_acc', 'margin'])
display(ranking)

heatmap_df = paired_quant.pivot(index='dataset', columns='method', values='fidelity').sort_index()
plt.figure(figsize=(12, 8))
sns.heatmap(heatmap_df, annot=True, fmt='.3f', cmap='YlGnBu', cbar_kws={'label': 'Fidelity'})
plt.title('Per-dataset fidelity by method')
plt.tight_layout()
plt.show()


In [ ]:
display(Markdown('### Paired Statistical Comparisons'))

stat_rows = []
method_order = [m for m in ['execution_trace', 'aggregated_transitions', 'shap', 'ice', 'anchors', 'tree_path', 'lime'] if m in paired_quant['method'].unique()]
for metric in ['fidelity', 'local_acc', 'margin']:
    stat_rows.append(friedman_table(paired_quant[['dataset', 'method', metric]].dropna(), method_order, metric))
    for comparator in [m for m in method_order if m != 'execution_trace']:
        stat_rows.append(paired_wilcoxon(paired_quant[['dataset', 'method', metric]].dropna(), 'execution_trace', comparator, metric))

stat_df = pd.DataFrame(stat_rows)
display(stat_df)

display(Markdown('### Controlled DPG Comparison'))
dpg_paired = best_dpg_table[best_dpg_table['method'].isin(['execution_trace', 'aggregated_transitions'])]
dpg_delta = dpg_paired.pivot(index='dataset', columns='method', values=['fidelity', 'local_acc', 'margin', 'edge_precision', 'edge_recall', 'recombination', 'explanation_confidence'])
dpg_delta.columns = [f'{a}_{b}' for a, b in dpg_delta.columns]
for metric in ['fidelity', 'local_acc', 'margin', 'edge_precision', 'edge_recall', 'recombination', 'explanation_confidence']:
    if f'{metric}_execution_trace' in dpg_delta.columns and f'{metric}_aggregated_transitions' in dpg_delta.columns:
        dpg_delta[f'delta_{metric}'] = dpg_delta[f'{metric}_execution_trace'] - dpg_delta[f'{metric}_aggregated_transitions']
display(dpg_delta.reset_index().sort_values('delta_fidelity', ascending=False))


## 3. Structural Faithfulness And Error Analysis

In [ ]:
display(Markdown('### DPG Structural Metrics'))
structural_cols = ['dataset', 'graph_construction_mode', 'local_matches_model_rate', 'local_accuracy', 'avg_node_recall', 'avg_node_precision', 'avg_edge_recall', 'avg_edge_precision', 'avg_recombination_rate', 'avg_path_purity', 'avg_competitor_exposure', 'avg_support_margin', 'avg_explanation_confidence', 'avg_critical_split_depth', 'avg_critical_node_contrast']
display(best_configs[structural_cols].sort_values(['graph_construction_mode', 'dataset']).reset_index(drop=True))

display(Markdown('### Cohort Summary'))
display(best_cohort_summary.sort_values(['graph_construction_mode', 'cohort_label']).reset_index(drop=True))

display(Markdown('### Agreement Summary'))
agreement_sort_col = 'agreement_group' if 'agreement_group' in best_agreement_summary.columns else 'disagree_with_model'
display(best_agreement_summary.sort_values(['graph_construction_mode', agreement_sort_col]).reset_index(drop=True))

display(Markdown('### Cohort Effect Sizes'))
display(cohort_effect_sizes.sort_values(['comparison', 'metric', 'graph_construction_mode']).reset_index(drop=True))

if not best_per_sample.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.boxplot(data=best_per_sample, x='graph_construction_mode', y='edge_precision', ax=axes[0])
    axes[0].set_title('Edge precision by DPG mode')
    sns.boxplot(data=best_per_sample, x='graph_construction_mode', y='explanation_confidence', ax=axes[1])
    axes[1].set_title('Explanation confidence by DPG mode')
    plt.tight_layout()
    plt.show()


## 4. Semantic Faithfulness And Critical-Node Experiments

In [ ]:
if semantic_summary.empty:
    display(Markdown('Semantic-faithfulness summary is not yet available as a full merged table. The focused case report is shown below if present.'))
else:
    display(Markdown('### Semantic Faithfulness Summary'))
    display(semantic_summary.sort_values(['method_family', 'mean_sufficiency_prob', 'mean_comprehensiveness_drop'], ascending=[True, False, False]).reset_index(drop=True))
    
    if not critical_branch_flip_summary.empty:
        display(Markdown('### Critical Branch Flip Summary'))
        display(critical_branch_flip_summary.sort_values(['dataset', 'method']).reset_index(drop=True))

display(Markdown('### Focused Critical Cases'))
if critical_case_reports.empty:
    display(Markdown('No focused critical case report found.'))
else:
    display(critical_case_reports.sort_values(['dataset', 'method_family', 'method']).reset_index(drop=True))

    for dataset, sample_idx in [('spambase', 859), ('vehicle', 56)]:
        subset = critical_case_reports[(critical_case_reports['dataset'] == dataset) & (critical_case_reports['sample_idx'] == sample_idx)]
        if subset.empty:
            continue
        display(Markdown(f'#### {dataset} sample {sample_idx}'))
        display(subset[['dataset', 'method_family', 'method', 'y_true', 'y_model_pred', 'y_explained_class', 'feature_count', 'sufficiency_prob', 'comprehensiveness_drop', 'critical_node_label', 'critical_branch_margin_delta', 'critical_flip_to_competitor']].reset_index(drop=True))


## 5. Phase C Case Studies And Paper Figures

In [ ]:
if not final_paper_subset.empty:
    display(Markdown('### Final Paper Subset'))
    display(final_paper_subset.reset_index(drop=True))

paper_cases = case_study_metadata.copy()
if not final_paper_subset.empty:
    paper_cases = paper_cases.merge(final_paper_subset[['case_label', 'dataset', 'sample_idx', 'graph_construction_mode']], on=['case_label', 'dataset', 'sample_idx', 'graph_construction_mode'], how='inner')

display(Markdown('### Case Study Metadata'))
display(paper_cases[['case_label', 'dataset', 'sample_idx', 'graph_construction_mode', 'true_label', 'model_pred', 'local_pred', 'explanation_confidence', 'support_margin', 'competitor_exposure', 'critical_node_label', 'critical_split_depth', 'critical_node_contrast', 'aggregate_png']].sort_values(['case_label', 'graph_construction_mode']).reset_index(drop=True))

for _, row in paper_cases.sort_values(['case_label', 'graph_construction_mode']).iterrows():
    title = f"{row['case_label']} | {row['dataset']} | sample {row['sample_idx']} | {row['graph_construction_mode']}"
    display(Markdown(f'#### {title}'))
    metrics = row[['true_label', 'model_pred', 'local_pred', 'explanation_confidence', 'support_margin', 'competitor_exposure', 'critical_node_label', 'critical_split_depth', 'critical_node_contrast']].to_frame().T
    display(metrics)
    for img_col, label in [('aggregate_png', 'Aggregate local subgraph'), ('pred_path_png', 'Predicted path'), ('competitor_path_png', 'Competitor path')]:
        path = row.get(img_col)
        if isinstance(path, str) and path and Path(path).exists():
            display(Markdown(label))
            display(Image(filename=str(path), width=1200))


## 6. Claims Checklist

In [ ]:
claims = []

def mean_of(df: pd.DataFrame, method: str, col: str) -> float:
    sub = df[df['method'] == method]
    if sub.empty or col not in sub.columns:
        return float('nan')
    return float(sub[col].mean())


paired_claim_df = paired_quant.copy()
exec_fid = mean_of(paired_claim_df, 'execution_trace', 'fidelity')
agg_fid = mean_of(paired_claim_df, 'aggregated_transitions', 'fidelity')
exec_acc = mean_of(paired_claim_df, 'execution_trace', 'local_acc')
agg_acc = mean_of(paired_claim_df, 'aggregated_transitions', 'local_acc')
exec_margin = mean_of(paired_claim_df, 'execution_trace', 'margin')
agg_margin = mean_of(paired_claim_df, 'aggregated_transitions', 'margin')

if exec_fid == exec_fid and agg_fid == agg_fid:
    claims.append(f'- Execution-trace DPG has higher mean fidelity than aggregated DPG: {exec_fid:.4f} vs {agg_fid:.4f}.')
if exec_acc == exec_acc and agg_acc == agg_acc:
    claims.append(f'- Execution-trace DPG has higher mean local accuracy than aggregated DPG: {exec_acc:.4f} vs {agg_acc:.4f}.')
if exec_margin == exec_margin and agg_margin == agg_margin:
    claims.append(f'- Execution-trace DPG has higher mean margin than aggregated DPG: {exec_margin:.4f} vs {agg_margin:.4f}.')

if not comparison_summary.empty:
    top_fidelity = comparison_summary.sort_values('mean_fidelity', ascending=False).iloc[0]
    claims.append(f"- Highest mean raw fidelity in the consolidated comparison is {top_fidelity['comparison_method']} ({top_fidelity['mean_fidelity']:.4f}).")

dpg_struct = best_configs.groupby('graph_construction_mode', as_index=False)[['avg_edge_precision', 'avg_edge_recall', 'avg_recombination_rate', 'avg_explanation_confidence']].mean(numeric_only=True)
if not dpg_struct.empty:
    claims.append('- Structural metrics should be interpreted only for DPG variants; attribution baselines do not expose the same path-level objects.')
    for _, row in dpg_struct.iterrows():
        claims.append(f"- {row['graph_construction_mode']}: edge precision={row['avg_edge_precision']:.4f}, edge recall={row['avg_edge_recall']:.4f}, recombination={row['avg_recombination_rate']:.4f}, explanation confidence={row['avg_explanation_confidence']:.4f}.")

if not critical_case_reports.empty:
    vehicle_case = critical_case_reports[(critical_case_reports['dataset'] == 'vehicle') & (critical_case_reports['sample_idx'] == 56) & (critical_case_reports['method'] == 'execution_trace')]
    if not vehicle_case.empty:
        r = vehicle_case.iloc[0]
        claims.append(f"- Vehicle sample 56 remains a strong recovery/disagreement case: critical node='{r['critical_node_label']}', branch flip to competitor={bool(r['critical_flip_to_competitor'])}.")
    sp_case = critical_case_reports[(critical_case_reports['dataset'] == 'spambase') & (critical_case_reports['sample_idx'] == 859) & (critical_case_reports['method'] == 'execution_trace')]
    if not sp_case.empty:
        r = sp_case.iloc[0]
        claims.append(f"- Spambase sample 859 is a strong error case, but under the stricter non-root rule its critical node is {r['critical_node_label']}. It should not be presented as the main critical-node example.")

display(Markdown('\n'.join(claims)))
